In [78]:
import sqlite3
import pandas as pd

In [79]:
#read and push csv file into DB
def read_and_push_csv_to_db(customers,transaction,db_name):
    conn = sqlite3.connect(db_name)
    df1 = pd.read_csv(customers)
    df1.to_sql("customers",conn, if_exists= "replace" , index=False)

    df2 = pd.read_csv(transaction)
    df2.to_sql("transactions", conn, if_exists = "replace", index=False)

    conn.close()
    

In [80]:
open("customers.csv","r").close()
open("transactions.csv","r").close()
read_and_push_csv_to_db("customers.csv","transactions.csv","ecommerce.db")



In [81]:
conn = sqlite3.connect("ecommerce.db")

sql_query = """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table';
            """
tables = pd.read_sql(sql_query, conn)
#create dataframe for each table
for table_name in tables["name"]:
    df = pd.read_sql(f'SELECT * FROM "{table_name}"' , conn)
    globals()[f"df_{table_name}"] = df
    print(f"Create dataframe: df_{table_name}")

conn.close()

Create dataframe: df_customers
Create dataframe: df_transactions


In [82]:
#print table name and column names

conn = sqlite3.connect("ecommerce.db")

for table_name in tables["name"]:
    print(f"\n Table Name: {table_name}")

    columns_query= f'PRAGMA table_info("{table_name}")'
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns["name"].tolist())

conn.close()


 Table Name: customers
Columns:
['customer_id', 'name', 'age', 'gender', 'state', 'signup_date', 'email', 'phone_number', 'subscribe']

 Table Name: transactions
Columns:
['transaction_id', 'customer_id', 'transaction_date', 'product_id', 'quantity', 'unit_price', 'payment_method', 'discount_applied', 'transaction_status', 'review_text']


In [83]:
# data cleaning 

In [84]:
df_customers.head()

,customer_id,name,age,gender,state,signup_date,email,phone_number,subscribe
0,CUST0001,Gandi,26.0,Male,Sulawesi Utara,2025-05-05,gandi_96@example.com,8422757160,No
1,CUST0002,Cecep,49.0,Male,Aceh,2025-06-02,cecep467@example.com,88171122035,No
2,CUST0003,Artawan,26.0,Male,Jambi,2025-02-11,wahyuninovi@example.net,8692932468,Yes
3,CUST0004,Mulya,61.0,Male,Nusa Tenggara Timur,2025-02-19,nnapitupulu@example.com,8515735210,Yes
4,CUST0005,Taufik,43.0,Male,Kepulauan Riau,2025-08-04,taufik_87@example.com,81034538051,Yes


In [85]:
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   customer_id   1010 non-null   object 
 1   name          1010 non-null   object 
 2   age           954 non-null    float64
 3   gender        1010 non-null   object 
 4   state         956 non-null    object 
 5   signup_date   1010 non-null   object 
 6   email         1010 non-null   object 
 7   phone_number  1010 non-null   int64  
 8   subscribe     1010 non-null   object 
dtypes: float64(1), int64(1), object(7)
memory usage: 71.1+ KB


In [86]:
# data cleaning of Customers
# 1. change name to customer_name
# 2. change datatype of signup_date
# 3. check duplication in customer_id and droping it


In [87]:
# 1. change name to coustomer_name

df_customers.rename( columns = {'name' : 'customer_name'} , inplace= True)
df_customers

,customer_id,customer_name,age,gender,state,signup_date,email,phone_number,subscribe
0,CUST0001,Gandi,26.0,Male,Sulawesi Utara,2025-05-05,gandi_96@example.com,8422757160,No
1,CUST0002,Cecep,49.0,Male,Aceh,2025-06-02,cecep467@example.com,88171122035,No
2,CUST0003,Artawan,26.0,Male,Jambi,2025-02-11,wahyuninovi@example.net,8692932468,Yes
3,CUST0004,Mulya,61.0,Male,Nusa Tenggara Timur,2025-02-19,nnapitupulu@example.com,8515735210,Yes
4,CUST0005,Taufik,43.0,Male,Kepulauan Riau,2025-08-04,taufik_87@example.com,81034538051,Yes
...,...,...,...,...,...,...,...,...,...
1005,CUST0996,Vinsen,47.0,Male,Bengkulu,2024-06-09,ibrani64@example.org,8585892932,Yes
1006,CUST0997,Ulya,59.0,Female,Sulawesi Utara,2024-06-23,putri15@example.com,81976792782,Yes
1007,CUST0998,Dina,41.0,Female,DKI Jakarta,2025-01-01,dina912@example.com,85499775196,Yes
1008,CUST0999,Yahya,63.0,Male,Kalimantan Timur,2023-12-12,chartati@example.net,8912787724,No


In [88]:
# 2. change datatype of signup_date
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'])

In [89]:
# 3. check duplication in customer_id and droping it

df_customers[df_customers.duplicated(subset = ['customer_id'], keep= False)]

,customer_id,customer_name,age,gender,state,signup_date,email,phone_number,subscribe
20,CUST0514,Adinata,60.0,Male,Aceh,2023-12-11,bgunawan@example.net,8287301073,Yes
72,CUST0679,Jayadi,46.0,Male,Kepulauan Bangka Belitung,2025-09-20,998jayadi@example.com,84642406422,No
104,CUST0522,Lurhur,34.0,Male,Papua,2024-08-14,andrianidimas@example.org,8739118659,Yes
108,CUST0412,Tira,65.0,Female,Riau,2024-12-22,tira_61@example.com,840843370,No
121,CUST0137,Faizah,58.0,Female,Sulawesi Tenggara,2025-02-15,faizah137@example.com,824030244596,Yes
141,CUST0137,Faizah,58.0,Female,Sulawesi Tenggara,2025-02-15,faizah137@example.com,824030244596,Yes
274,CUST0661,Puji,32.0,Female,DI Yogyakarta,2025-09-05,puji_62@example.com,82263520270,Yes
417,CUST0412,Tira,65.0,Female,Riau,2024-12-22,tira_61@example.com,840843370,No
440,CUST0738,Tugiman,25.0,Male,Banten,2025-02-08,tugiman_3@example.com,8928576444,Yes
520,CUST0514,Adinata,60.0,Male,Aceh,2023-12-11,bgunawan@example.net,8287301073,Yes


In [90]:
#droping duplication
df_customers = df_customers.drop_duplicates(subset=['customer_id'], keep='first')

In [91]:
#data cleaning of transactions

In [92]:
#df_transactions.head()
df_transactions.tail()

,transaction_id,customer_id,transaction_date,product_id,quantity,unit_price,payment_method,discount_applied,transaction_status,review_text
8194,TX_0000005735,CUST0695,2025-09-24,GA-889,31,132000,Debit Card,24.0,Completed,Very satisfied with this purchase
8195,TX_0000005192,CUST0627,2023-12-23,SM-851,40,377000,Cash,3.0,Completed,None
8196,TX_0000005391,CUST0655,2024-10-10,SV-067,2,83000,Debit Card,42.0,Completed,"Item damaged upon arrival, disappointed"
8197,TX_0000000861,CUST0107,2024-11-16,IR-720,23,112000,Debit Card,48.0,Completed,"Honest seller, item as described"
8198,TX_0000007271,CUST0882,2025-02-08,ES-840,26,261000,Bank Transfer,31.0,Completed,Bit overpriced but quality is okay


In [93]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8199 entries, 0 to 8198
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   transaction_id      8199 non-null   object 
 1   customer_id         8199 non-null   object 
 2   transaction_date    8199 non-null   object 
 3   product_id          8199 non-null   object 
 4   quantity            8199 non-null   int64  
 5   unit_price          8199 non-null   int64  
 6   payment_method      8199 non-null   object 
 7   discount_applied    7773 non-null   float64
 8   transaction_status  8199 non-null   object 
 9   review_text         3283 non-null   object 
dtypes: float64(1), int64(2), object(7)
memory usage: 640.7+ KB


In [94]:
# 1. Change the datatype of transaction_date
# 2. check duplication of transaction_id
# 3. missing value of discount_applied

In [95]:
# 1. Change the datatype of transaction_date

df_transactions['transaction_date'] = pd.to_datetime(df_transactions['transaction_date'])

In [96]:
# 2. check duplication of transaction_id
df_transactions[df_transactions.duplicated(subset = ['transaction_id'], keep =False)]

,transaction_id,customer_id,transaction_date,product_id,quantity,unit_price,payment_method,discount_applied,transaction_status,review_text


In [97]:
df_transactions['transaction_status'].unique()

array(['Completed', 'Pending', 'Cancelled', 'Refunded'], dtype=object)

In [98]:
df_transactions['payment_method'].unique()

array(['Credit Card', 'E-wallet', 'Cash', 'Bank Transfer', 'Debit Card'],
      dtype=object)

In [99]:
df_transactions['product_id'].unique()

array(['WZ-802', 'LE-969', 'CR-562', ..., 'SV-067', 'IR-720', 'ES-840'],
      dtype=object)

In [100]:
# 3. missing value of discount_applied and ,if it is related to cancellation/refund
df_transactions['discount_missing'] = df_transactions['discount_applied'].isnull()
df_transactions.groupby('discount_missing')['transaction_status'].value_counts(normalize = True)

discount_missing  transaction_status
False             Completed             0.851280
                  Cancelled             0.079249
                  Refunded              0.046314
                  Pending               0.023157
True              Completed             0.842723
                  Cancelled             0.105634
                  Refunded              0.037559
                  Pending               0.014085
Name: proportion, dtype: float64

In [101]:
df_transactions['discount_missing'].value_counts()

discount_missing
False    7773
True      426
Name: count, dtype: int64

In [102]:
df_transactions['discount_missing_flag'] = df_transactions['discount_applied'].isnull()

In [103]:
df_transactions['discount_missing_flag'].sum()

426

In [104]:
df_transactions['discount_applied'] = df_transactions['discount_applied'].fillna(0)

In [105]:
df_transactions['discount_applied']

0        6.0
1        7.0
2        8.0
3       42.0
4       27.0
        ... 
8194    24.0
8195     3.0
8196    42.0
8197    48.0
8198    31.0
Name: discount_applied, Length: 8199, dtype: float64

In [106]:
df_transactions['discount_applied'].isnull().sum()

0

In [107]:
# missing value of age if it is related to cancellation/refund

In [110]:

df_customers.loc[:, 'age_missing'] = df_customers['age'].isnull()

In [111]:
merged = df_customers.merge(df_transactions, on='customer_id', how='inner')
print(merged.shape)

(8199, 21)


In [112]:
merged.groupby('age_missing')['transaction_status'].value_counts(normalize=True)

age_missing  transaction_status
False        Completed             0.851069
             Cancelled             0.080493
             Refunded              0.045755
             Pending               0.022683
True         Completed             0.847107
             Cancelled             0.082645
             Refunded              0.047521
             Pending               0.022727
Name: proportion, dtype: float64

In [113]:
median_age = df_customers['age'].median()
df_customers.loc[:,'age'] = df_customers.fillna(median_age)

In [114]:
df_customers['age'].isnull().sum()

0

In [115]:
print(median_age)

42.0


In [116]:
#checking with review_text if it is related to cancellation/refund

In [117]:
df_transactions['review_text'].isnull().sum()

4916

In [118]:
df_transactions['review_text'].isnull().mean()

0.5995853152823515

In [119]:
df_transactions.loc[:, 'has_review'] = df_transactions['review_text'].notnull()

In [120]:
df_transactions.loc[:,'review_text'] = df_transactions['review_text'].fillna('')

In [121]:
df_transactions['review_text'].isnull().sum()

0

In [122]:
df_transactions.groupby('has_review')['transaction_status'].value_counts(normalize=True)

has_review  transaction_status
False       Completed             0.850488
            Cancelled             0.078723
            Refunded              0.048413
            Pending               0.022376
True        Completed             0.851355
            Cancelled             0.083460
            Refunded              0.042035
            Pending               0.023150
Name: proportion, dtype: float64

In [123]:
#checking state and, if it is related to cancellation/refund

In [130]:
df_customers.loc[:,'state_missing'] = df_customers['state'].isnull()

In [125]:
merged = df_customers.merge(df_transactions , on ='customer_id' , how = 'inner' )

In [126]:
print(merged.shape)

(8199, 23)


In [127]:
merged.groupby('state_missing')['transaction_status'].value_counts(normalize=True)

state_missing  transaction_status
False          Completed             0.851909
               Cancelled             0.079979
               Refunded              0.045279
               Pending               0.022833
True           Completed             0.832215
               Cancelled             0.091723
               Refunded              0.055928
               Pending               0.020134
Name: proportion, dtype: float64

In [128]:
merged['state_missing'].value_counts()

state_missing
False    7752
True      447
Name: count, dtype: int64

In [134]:
df_customers['state'].isnull().sum()

0

In [132]:
df_customers.loc[:, 'state'] = df_customers['state'].fillna('Unknown')

In [135]:
df_customers.to_csv('customers_clean.csv', index=False)

In [136]:
df_transactions.to_csv('transactions_clean.csv', index=False)